# Expanded XAI Evaluation
## Full CheXlocalize Frontal Set (413 images) + 400 ChestX-ray14 Test Images

**Methods:** LayerCAM · FPN-LayerCAM · ScoreCAM · FPN-ScoreCAM · LIME  
**FPN config (3-level, original):** denseblock1 (w=0.20) · denseblock2 (w=0.35) · denseblock4 (w=0.45)  
**CheXlocalize:** all 5 metrics (PG, IoU@0.5, mIoU, Deletion AUC, Insertion AUC)  
**ChestX-ray14:** Deletion AUC + Insertion AUC only (no GT masks)  

---
### Strategy for 400 ChestX-ray14 images
Merge existing 100-image sample with 300 new stratified images → 400 total.  
Old result CSV (`results_chestxray14.csv`) is loaded first so already-evaluated  
images are skipped — no redundant computation.

### Estimated GPU time
- LayerCAM + FPN-LayerCAM + LIME: ~2–4 hours (813 images)
- ScoreCAM + FPN-ScoreCAM: ~12–18 hours (1664 + 3712 passes/class/image)

## 0. Imports & Reproducibility

In [ ]:
import gc, json, random, shutil, warnings
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.segmentation import slic
from sklearn.linear_model import Ridge
from sklearn.preprocessing import normalize as sk_normalise
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
CHEXPERT_CKPT         = Path('./chexpert_output/best_model.pth')

# ChestX-ray14 paths
NIH_DATASET_ROOT      = Path('/path/to/ChestX-ray14')   # <-- SET THIS
NIH_CSV_PATH          = NIH_DATASET_ROOT / 'Data_Entry_2017_v2020.csv'
NIH_TEST_LIST         = NIH_DATASET_ROOT / 'test_list.txt'   # official test split
NIH_OLD_MANIFEST      = Path('./test_samples/manifest.csv')  # existing 100 images
NIH_OLD_RESULTS       = Path('./xai_evaluation/results_chestxray14.csv')  # prior results
NIH_400_DIR           = Path('./test_samples_400')           # merged 400-image dir

# CheXlocalize paths
CHEXLOCALIZE_ROOT     = Path('/path/to/CheXlocalize')    # <-- SET THIS
CHEX_TEST_CSV         = CHEXLOCALIZE_ROOT / 'test_labels.csv'
GT_ANNOTATIONS_PATH   = Path('./gt_annotations_test.json')

# Output
EVAL_OUTPUT_DIR       = Path('./xai_evaluation_full')
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(EVAL_OUTPUT_DIR / 'chestxray14').mkdir(exist_ok=True)
(EVAL_OUTPUT_DIR / 'chexlocalize').mkdir(exist_ok=True)

# ── CheXpert label space (model output) ────────────────────────────────────────
LABELS: List[str] = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
    'Atelectasis', 'Pneumothorax', 'Effusion',
    'Pleural Other', 'Fracture', 'Support Devices', 'No Finding',
]
NUM_CLASSES = len(LABELS)

# ── NIH label set (for ground-truth lookup in merged manifest) ─────────────────
NIH_LABELS: List[str] = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax',
]

# ── CheXlocalize GT label mapping ──────────────────────────────────────────────
LABEL_MAP: Dict[str, str] = {
    'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum',
    'Cardiomegaly'              : 'Cardiomegaly',
    'Lung Opacity'              : 'Airspace Opacity',
    'Lung Lesion'               : 'Lung Lesion',
    'Edema'                     : 'Edema',
    'Consolidation'             : 'Consolidation',
    'Atelectasis'               : 'Atelectasis',
    'Pneumothorax'              : 'Pneumothorax',
    'Effusion'                  : 'Pleural Effusion',
    'Support Devices'           : 'Support Devices',
}

# ── FPN config (original 3-level) ──────────────────────────────────────────────
FPN_LAYERS: List[Tuple[str, float]] = [
    ('features.denseblock1', 0.20),
    ('features.denseblock2', 0.35),
    ('features.denseblock4', 0.45),
]

# ── Hyperparameters ────────────────────────────────────────────────────────────
IMG_SIZE        = 224
THRESHOLD       = 0.5
SCORECAM_BATCH  = 16
LIME_N_SEGMENTS = 80
LIME_COMPACTNESS= 10
LIME_N_SAMPLES  = 256
LIME_TOP_FEAT   = 10
DEL_INS_STEPS   = 50
DEL_INS_BATCH   = 20
BLUR_KERNEL     = 51
IMAGENET_MEAN   = [0.485, 0.456, 0.406]
IMAGENET_STD    = [0.229, 0.224, 0.225]
XAI_METHODS     = ['LayerCAM', 'FPN-LayerCAM', 'ScoreCAM', 'FPN-ScoreCAM', 'LIME']

print('Configuration ready.')

## 2. Architecture Verification (programmatic)

In [ ]:
def verify_densenet169_architecture() -> Dict[str, dict]:
    """
    Runs a dummy forward pass and records output shape for each dense block.
    Confirms spatial dimensions, channel counts, and effective stride.
    """
    _model = models.densenet169(weights=None).eval()
    shapes: Dict[str, tuple] = {}
    hooks  = []
    for blk in ['denseblock1','denseblock2','denseblock3','denseblock4']:
        layer = getattr(_model.features, blk)
        h = layer.register_forward_hook(
            lambda m, i, o, name=blk: shapes.update({name: tuple(o.shape)})
        )
        hooks.append(h)

    with torch.no_grad():
        _model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE))
    for h in hooks:
        h.remove()
    del _model

    print(f'DenseNet169 architecture (input {IMG_SIZE}×{IMG_SIZE}):')
    print(f'{"Block":<16}  {"Spatial":>8}  {"Channels":>10}  {"Stride":>8}')
    print('-' * 48)
    strides = {'denseblock1': 4, 'denseblock2': 8, 'denseblock3': 16, 'denseblock4': 32}
    info = {}
    for blk, shape in shapes.items():
        _, C, H, W = shape
        s = strides[blk]
        print(f'{blk:<16}  {H}×{W:>3}        {C:>10}  ×{s:>7}')
        info[blk] = {'channels': C, 'spatial': (H, W), 'stride': s}
    return info


ARCH_INFO = verify_densenet169_architecture()

## 3. Model Loading

In [ ]:
def load_model(ckpt_path: Path) -> nn.Module:
    model = models.densenet169(weights=None)
    in_feat = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_feat, NUM_CLASSES),
    )
    ckpt  = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt.get('state_dict', ckpt), strict=True)
    model.eval()
    print(f'Loaded checkpoint — epoch {ckpt.get("epoch","?")}, '
          f'val AUC {ckpt.get("val_auc","?")}')
    return model


model     = load_model(CHEXPERT_CKPT).to(DEVICE)
model_cpu = deepcopy(model).to('cpu').eval()

## 4. Build 400-Image ChestX-ray14 Sample

In [ ]:
def build_image_index(root: Path) -> Dict[str, Path]:
    idx = {p.name: p for p in root.rglob('*.png')}
    print(f'Indexed {len(idx):,} images under {root}')
    return idx


def make_unique_nih_filename(image_index_value: str) -> str:
    """NIH images already have unique filenames (00000001_001.png). Pass through."""
    return image_index_value


def build_400_nih_sample(
    old_manifest_path : Path,
    csv_path          : Path,
    image_index       : Dict[str, Path],
    output_dir        : Path,
    n_extra           : int = 300,
    seed              : int = SEED,
) -> pd.DataFrame:
    """
    Merges the existing 100-image NIH manifest with 300 new stratified images.

    Strategy:
      1. Load old manifest → copy images already present to output_dir.
      2. Load full test set CSV, drop images already in old manifest.
      3. Stratified sample 300 new images (at least 1 positive per NIH label
         not already covered), fill remainder randomly.
      4. Copy new images to output_dir with their original filename.
      5. Write merged manifest.csv.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)

    # ── Load old 100 manifest ─────────────────────────────────────────────────
    old_mf  = pd.read_csv(old_manifest_path)
    old_fns = set(old_mf['filename'].tolist())
    print(f'Old manifest: {len(old_mf)} images')

    # Copy old images to new output dir
    for fn in old_fns:
        src = old_manifest_path.parent / fn
        dst = output_dir / fn
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

    # ── Load full NIH test CSV ────────────────────────────────────────────────
    df = pd.read_csv(csv_path)

    # Filter to test set using official test list (if available)
    if NIH_TEST_LIST.exists():
        test_files = set(Path(NIH_TEST_LIST).read_text().strip().split('\n'))
        df = df[df['Image Index'].isin(test_files)].reset_index(drop=True)
    else:
        print('WARNING: test_list.txt not found — sampling from full CSV.')

    # Rename columns
    df.rename(columns={'Image Index': 'image_index',
                       'Finding Labels': 'finding_labels',
                       'Patient ID': 'patient_id'}, inplace=True)

    # One-hot encode NIH labels
    for lbl in NIH_LABELS:
        df[lbl] = df['finding_labels'].str.contains(lbl, regex=False).astype(np.float32)

    # Keep only images present on disk and not already sampled
    df = df[df['image_index'].isin(image_index)].reset_index(drop=True)
    df = df[~df['image_index'].isin(old_fns)].reset_index(drop=True)
    print(f'Pool for new sampling: {len(df):,} images')

    # ── Stratified sample ─────────────────────────────────────────────────────
    selected: set = set()
    for lbl in NIH_LABELS:
        pos = df.index[df[lbl] == 1].tolist()
        if pos:
            selected.add(int(rng.choice(pos)))

    remaining = list(set(df.index) - selected)
    rng.shuffle(remaining)
    selected.update(remaining[: n_extra - len(selected)])
    new_df = df.iloc[sorted(selected)].reset_index(drop=True)

    # ── Copy new images ───────────────────────────────────────────────────────
    new_rows = []
    failed   = []
    for _, row in new_df.iterrows():
        fn  = row['image_index']
        src = image_index[fn]
        dst = output_dir / fn
        if not src.exists():
            failed.append(fn); continue
        shutil.copy2(src, dst)
        new_rows.append({
            'filename'      : fn,
            'source_path'   : str(src),
            'finding_labels': row['finding_labels'],
            **{lbl: int(row[lbl]) for lbl in NIH_LABELS},
        })

    new_mf = pd.DataFrame(new_rows)

    # ── Merge old + new manifests ─────────────────────────────────────────────
    # Align columns: add missing NIH label columns to old manifest
    for lbl in NIH_LABELS:
        if lbl not in old_mf.columns:
            old_mf[lbl] = 0
    if 'finding_labels' not in old_mf.columns:
        old_mf['finding_labels'] = 'Unknown'

    merged = pd.concat([old_mf, new_mf], ignore_index=True)
    merged.drop_duplicates(subset='filename', keep='first', inplace=True)
    merged.reset_index(drop=True, inplace=True)
    merged.to_csv(output_dir / 'manifest.csv', index=False)

    print(f'Merged manifest: {len(merged)} images ({len(old_mf)} old + {len(new_mf)} new)')
    if failed:
        print(f'WARNING: {len(failed)} images not found on disk')
    return merged


nih_image_index = build_image_index(NIH_DATASET_ROOT / 'images')
nih_manifest_400 = build_400_nih_sample(
    old_manifest_path = NIH_OLD_MANIFEST,
    csv_path          = NIH_CSV_PATH,
    image_index       = nih_image_index,
    output_dir        = NIH_400_DIR,
)

## 5. Build Full CheXlocalize Frontal Sample (413 images)

In [ ]:
def load_chexlocalize_full(
    test_csv_path : Path,
    chex_root     : Path,
    output_dir    : Path,
    seed          : int = SEED,
) -> pd.DataFrame:
    """
    Loads all frontal-view images from CheXlocalize test_labels.csv,
    copies them to output_dir with collision-free filenames
    (patient_study_view.jpg), and writes manifest.csv.

    Frontal filter: keeps only rows where the Path contains
    'frontal' (view1_frontal, view2_frontal, etc.).
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(test_csv_path)

    # Frontal filter
    before = len(df)
    df = df[df['Path'].str.contains('frontal', case=False)].reset_index(drop=True)
    print(f'Frontal filter: {before} → {len(df)} images')

    # Apply uncertainty policy: -1 → 0, NaN → 0
    chex_label_cols = [c for c in df.columns if c not in ['Path']]
    for col in chex_label_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).replace(-1.0, 0.0).astype(np.float32)

    def resolve_path(csv_path_val: str) -> Path:
        p = chex_root / csv_path_val
        if p.exists(): return p
        # Try stripping CheXpert-v1.0/ prefix
        stripped = csv_path_val.replace('CheXpert-v1.0/', '', 1)
        p2 = chex_root / stripped
        return p2

    def make_unique_fn(csv_path_val: str) -> str:
        parts = Path(csv_path_val).parts
        # patient / study / view.jpg  →  patientXXX_studyY_viewZ.jpg
        return '_'.join(parts[-3:]) if len(parts) >= 3 else parts[-1]

    rows   = []
    failed = []
    seen   : set = set()

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Copy CheXlocalize'):
        src = resolve_path(row['Path'])
        fn  = make_unique_fn(row['Path'])
        if fn in seen:
            stem, ext = fn.rsplit('.', 1)
            fn = f'{stem}_{len(seen)}.{ext}'
        seen.add(fn)
        dst = output_dir / fn

        if not src.exists():
            failed.append(str(src)); continue
        if not dst.exists():
            shutil.copy2(src, dst)

        label_vals = {
            lbl: float(row[lbl]) if lbl in row.index else 0.0
            for lbl in LABELS
        }
        rows.append({
            'filename'      : fn,
            'source_path'   : str(src),
            'original_path' : row['Path'],
            **label_vals,
        })

    manifest = pd.DataFrame(rows)
    manifest.to_csv(output_dir / 'manifest.csv', index=False)
    print(f'Saved {len(manifest)} frontal CheXlocalize images')
    if failed:
        print(f'WARNING: {len(failed)} images not found on disk')
    return manifest


CHEX_413_DIR     = EVAL_OUTPUT_DIR / 'chexlocalize_images'
chex_manifest_413 = load_chexlocalize_full(
    test_csv_path = CHEX_TEST_CSV,
    chex_root     = CHEXLOCALIZE_ROOT,
    output_dir    = CHEX_413_DIR,
)
print(f'CheXlocalize frontal set: {len(chex_manifest_413)} images')

## 6. GT Mask Loader

In [ ]:
with open(GT_ANNOTATIONS_PATH) as f:
    GT_ANNOTATIONS: Dict = json.load(f)
print(f'GT annotations: {len(GT_ANNOTATIONS)} images')


def rasterise_polygons(img_key: str, gt_label: str,
                        out_size: int = IMG_SIZE) -> Optional[np.ndarray]:
    if img_key not in GT_ANNOTATIONS:
        return None
    entry = GT_ANNOTATIONS[img_key]
    if gt_label not in entry:
        return None
    H, W  = entry['img_size']
    mask  = np.zeros((H, W), dtype=np.uint8)
    for poly in entry[gt_label]:
        pts = np.array(poly, dtype=np.float32)
        if len(pts) >= 3:
            cv2.fillPoly(mask, [pts.astype(np.int32).reshape(-1,1,2)], 1)
    if mask.sum() == 0:
        return None
    return cv2.resize(mask, (out_size, out_size),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)


def filename_to_gt_key(filename: str) -> str:
    return Path(filename).stem

## 7. Image Utilities

In [ ]:
PREPROCESS = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image(path: Path) -> Tuple[torch.Tensor, np.ndarray]:
    pil     = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_rgb = np.array(pil, dtype=np.uint8)
    tensor  = PREPROCESS(pil).unsqueeze(0).to(DEVICE)
    return tensor, img_rgb


def normalise_cam(cam: np.ndarray) -> np.ndarray:
    mn, mx = cam.min(), cam.max()
    if mx - mn < 1e-8:
        return np.zeros_like(cam, dtype=np.float32)
    return ((cam - mn) / (mx - mn)).astype(np.float32)


@torch.no_grad()
def get_probs(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    return model(tensor).sigmoid().squeeze(0).cpu().numpy()

## 8. XAI Methods

In [ ]:
class ActivationGradientHook:
    def __init__(self, layer):
        self.activation = self.gradient = None
        self._fwd = layer.register_forward_hook(
            lambda m,i,o: setattr(self,'activation',o.detach()))
        self._bwd = layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,'gradient',go[0].detach()))
    def remove(self):
        self._fwd.remove(); self._bwd.remove()


def get_layer(model, name):
    m = model
    for p in name.split('.'): m = getattr(m,p)
    return m


def layercam(model, tensor, class_idx, layer_name='features.denseblock4'):
    hook = ActivationGradientHook(get_layer(model, layer_name))
    model.zero_grad()
    model(tensor)[0, class_idx].backward()
    act = hook.activation.squeeze(0)
    grad = hook.gradient.squeeze(0)
    hook.remove()
    cam = F.relu((act * F.relu(grad)).sum(0)).cpu().numpy()
    return normalise_cam(cv2.resize(cam,(IMG_SIZE,IMG_SIZE),
                                    interpolation=cv2.INTER_LINEAR))


def fpn_layercam(model, tensor, class_idx,
                 fpn_layers=FPN_LAYERS):
    fused = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.float32)
    for layer_name, weight in fpn_layers:
        hook = ActivationGradientHook(get_layer(model, layer_name))
        model.zero_grad()
        model(tensor)[0, class_idx].backward(retain_graph=True)
        act = hook.activation.squeeze(0)
        grad = hook.gradient.squeeze(0)
        hook.remove()
        cam = F.relu((act * F.relu(grad)).sum(0)).cpu().numpy()
        cam = cv2.resize(cam,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
        fused += weight * normalise_cam(cam)
    model.zero_grad()
    return normalise_cam(fused)


def scorecam(model, tensor, class_idx, layer_name='features.denseblock4',
             batch_size=SCORECAM_BATCH):
    acts = {}
    h = get_layer(model,layer_name).register_forward_hook(
        lambda m,i,o: acts.update({'v':o.detach()}))
    with torch.no_grad(): model(tensor)
    h.remove()
    activations = acts['v'].squeeze(0)
    n_ch = activations.shape[0]
    with torch.no_grad():
        baseline = model(torch.zeros_like(tensor)).sigmoid()[0,class_idx].item()
    ch_w = np.zeros(n_ch, dtype=np.float32)
    for bs in range(0, n_ch, batch_size):
        be = min(bs+batch_size, n_ch)
        masked = []
        for c in range(bs, be):
            ch = activations[c].cpu().numpy()
            ch = cv2.resize(ch,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
            mn,mx = ch.min(),ch.max()
            ch = (ch-mn)/(mx-mn+1e-8)
            masked.append(tensor * torch.from_numpy(ch).float().to(DEVICE)[None,None])
        with torch.no_grad():
            s = model(torch.cat(masked)).sigmoid()[:,class_idx].cpu().numpy()
        ch_w[bs:be] = s - baseline
    wt  = torch.from_numpy(ch_w).to(activations.device)
    cam = F.relu((F.relu(wt)[:,None,None]*activations).sum(0)).cpu().numpy()
    return normalise_cam(cv2.resize(cam,(IMG_SIZE,IMG_SIZE),
                                    interpolation=cv2.INTER_LINEAR))


def fpn_scorecam(model, tensor, class_idx,
                 fpn_layers=FPN_LAYERS):
    fused = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.float32)
    for layer_name, weight in fpn_layers:
        fused += weight * scorecam(model,tensor,class_idx,layer_name=layer_name)
    return normalise_cam(fused)


def lime_cam(model_cpu_ref, img_rgb, class_idx):
    segs   = slic(img_rgb,n_segments=LIME_N_SEGMENTS,
                  compactness=LIME_COMPACTNESS,start_label=0)
    n_segs = segs.max()+1
    mean_c = img_rgb.mean(axis=(0,1))
    rng    = np.random.default_rng(SEED)
    Z      = rng.integers(0,2,size=(LIME_N_SAMPLES,n_segs)).astype(np.float32)
    Z[0,:] = 1.0
    perturbed = []
    for z in Z:
        pimg = img_rgb.copy().astype(np.float32)
        for sid in range(n_segs):
            if z[sid]==0: pimg[segs==sid]=mean_c
        perturbed.append(PREPROCESS(Image.fromarray(pimg.astype(np.uint8))))
    batch  = torch.stack(perturbed)
    scores = []
    with torch.no_grad():
        for i in range(0,len(batch),32):
            scores.append(model_cpu_ref(batch[i:i+32]).sigmoid())
    scores = torch.cat(scores).numpy()
    ones   = np.ones((1,n_segs),dtype=np.float32)
    wts    = np.clip((sk_normalise(Z,'l2')@sk_normalise(ones,'l2').T).squeeze(),0,None)
    ridge  = Ridge(alpha=1.0).fit(Z,scores[:,class_idx],sample_weight=wts)
    coefs  = np.maximum(ridge.coef_,0)
    top    = np.argsort(coefs)[-LIME_TOP_FEAT:]
    mc     = np.zeros_like(coefs); mc[top]=coefs[top]
    hmap   = np.zeros((IMG_SIZE,IMG_SIZE),dtype=np.float32)
    for sid in range(n_segs):
        hmap[segs==sid]=mc[sid]
    return normalise_cam(hmap)


def compute_all_cams(model, model_cpu_ref, tensor, img_rgb, class_idx):
    return {
        'LayerCAM'    : layercam(model,tensor,class_idx),
        'FPN-LayerCAM': fpn_layercam(model,tensor,class_idx),
        'ScoreCAM'    : scorecam(model,tensor,class_idx),
        'FPN-ScoreCAM': fpn_scorecam(model,tensor,class_idx),
        'LIME'        : lime_cam(model_cpu_ref,img_rgb,class_idx),
    }

## 9. Evaluation Metrics

In [ ]:
def pointing_game(cam, gt_mask):
    max_val  = cam.max()
    for r,c in np.argwhere(cam==max_val):
        if gt_mask[r,c]==1: return 1.0
    return 0.0


def iou_at_threshold(cam, gt_mask, t=0.5):
    pred = (cam>=t).astype(np.uint8)
    inter = (pred & gt_mask).sum()
    union = (pred | gt_mask).sum()
    return float(inter)/float(union) if union>0 else float('nan')


def mean_iou(cam, gt_mask,
             thresholds=np.linspace(0.1,0.9,9)):
    vals = [iou_at_threshold(cam,gt_mask,t) for t in thresholds]
    valid = [v for v in vals if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float('nan')


def _blur_baseline(tensor, k=BLUR_KERNEL):
    k = k|1
    return F.avg_pool2d(tensor,k,stride=1,padding=k//2,
                        count_include_pad=False).detach()


@torch.no_grad()
def deletion_insertion_auc(model, tensor, cam, class_idx,
                            n_steps=DEL_INS_STEPS, batch_size=DEL_INS_BATCH):
    H,W       = cam.shape
    order     = np.argsort(cam.ravel())[::-1]
    steps     = np.linspace(0,H*W,n_steps+1,dtype=int)
    mean_v    = tensor.mean(dim=(0,2,3),keepdim=True).expand_as(tensor).clone()
    blur_base = _blur_baseline(tensor)
    d_scores, i_scores = [], []
    bd, bi = [], []
    for n in steps:
        d = tensor.clone()
        iv = blur_base.clone()
        if n>0:
            r = order[:n]//W; c = order[:n]%W
            d[0,:,r,c] = mean_v[0,:,r,c]
            iv[0,:,r,c]= tensor[0,:,r,c]
        bd.append(d); bi.append(iv)
        if len(bd)==batch_size or n==steps[-1]:
            ds = model(torch.cat(bd)).sigmoid()[:,class_idx].cpu().numpy()
            is_ = model(torch.cat(bi)).sigmoid()[:,class_idx].cpu().numpy()
            d_scores.extend(ds.tolist()); i_scores.extend(is_.tolist())
            bd, bi = [], []
    xs = np.linspace(0,1,len(d_scores))
    return float(np.trapz(d_scores,xs)), float(np.trapz(i_scores,xs))

## 10. Per-Image Evaluation Core

In [ ]:
def evaluate_image(
    model, model_cpu_ref, img_path, img_key=None
):
    """
    Returns list of result dicts for one image.
    img_key: GT lookup key if CheXlocalize; None for ChestX-ray14.
    """
    tensor, img_rgb = load_image(img_path)
    probs = get_probs(model, tensor)
    pred_ids = [i for i,p in enumerate(probs) if p>=THRESHOLD] or [int(np.argmax(probs))]

    rows = []
    for class_idx in pred_ids:
        label = LABELS[class_idx]

        gt_mask = None
        if img_key and label in LABEL_MAP:
            gt_mask = rasterise_polygons(img_key, LABEL_MAP[label])

        cams = compute_all_cams(model, model_cpu_ref, tensor, img_rgb, class_idx)

        for method, cam in cams.items():
            pg   = pointing_game(cam,gt_mask)    if gt_mask is not None else float('nan')
            iou  = iou_at_threshold(cam,gt_mask) if gt_mask is not None else float('nan')
            miou = mean_iou(cam,gt_mask)          if gt_mask is not None else float('nan')
            d,i  = deletion_insertion_auc(model,tensor,cam,class_idx)
            rows.append({
                'filename'     : img_path.name,
                'img_key'      : img_key or '',
                'method'       : method,
                'label'        : label,
                'class_idx'    : class_idx,
                'prob'         : round(float(probs[class_idx]),4),
                'gt_available' : gt_mask is not None,
                'pointing_game': round(pg,4)   if not np.isnan(pg)   else float('nan'),
                'iou_05'       : round(iou,4)  if not np.isnan(iou)  else float('nan'),
                'miou'         : round(miou,4) if not np.isnan(miou) else float('nan'),
                'deletion_auc' : round(d,4),
                'insertion_auc': round(i,4),
            })
    torch.cuda.empty_cache(); gc.collect()
    return rows

## 11. Resume-Friendly Batch Runner

In [ ]:
def run_evaluation(
    model, model_cpu_ref,
    manifest_path, image_dir,
    dataset_name, has_gt_masks,
    results_path,
):
    manifest = pd.read_csv(manifest_path)

    if results_path.exists():
        existing   = pd.read_csv(results_path)
        done_files = set(existing['filename'].unique())
        print(f'Resuming — {len(done_files)} / {len(manifest)} images already done.')
    else:
        done_files = set()
        pd.DataFrame(columns=[
            'filename','img_key','method','label','class_idx','prob',
            'gt_available','pointing_game','iou_05','miou',
            'deletion_auc','insertion_auc'
        ]).to_csv(results_path, index=False)

    skipped = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest),
                       desc=f'Eval [{dataset_name}]'):
        fname    = row['filename']
        if fname in done_files: continue
        img_path = image_dir / fname
        if not img_path.exists():
            skipped.append(fname); continue

        img_key = (
            filename_to_gt_key(fname)
            if has_gt_masks else None
        )
        try:
            result_rows = evaluate_image(
                model, model_cpu_ref, img_path, img_key)
            pd.DataFrame(result_rows).to_csv(
                results_path, mode='a', header=False, index=False)
        except Exception as e:
            print(f'  ERROR {fname}: {e}')
            skipped.append(fname)

    print(f'{dataset_name}: done. Skipped: {len(skipped)}')
    return pd.read_csv(results_path)

## 12. Run — CheXlocalize (413 frontal images)

In [ ]:
chex_results = run_evaluation(
    model           = model,
    model_cpu_ref   = model_cpu,
    manifest_path   = CHEX_413_DIR / 'manifest.csv',
    image_dir       = CHEX_413_DIR,
    dataset_name    = 'CheXlocalize-413',
    has_gt_masks    = True,
    results_path    = EVAL_OUTPUT_DIR / 'results_chexlocalize_413.csv',
)
print(f'CheXlocalize results: {chex_results.shape}')

## 13. Run — ChestX-ray14 (400 images)

In [ ]:
# Load prior 100-image results to prepopulate the CSV
# so the runner skips those images automatically
results_400_path = EVAL_OUTPUT_DIR / 'results_chestxray14_400.csv'
if NIH_OLD_RESULTS.exists() and not results_400_path.exists():
    shutil.copy2(NIH_OLD_RESULTS, results_400_path)
    print(f'Seeded results file with {len(pd.read_csv(results_400_path))} '
          'prior rows from 100-image evaluation.')

nih_results = run_evaluation(
    model           = model,
    model_cpu_ref   = model_cpu,
    manifest_path   = NIH_400_DIR / 'manifest.csv',
    image_dir       = NIH_400_DIR,
    dataset_name    = 'ChestX-ray14-400',
    has_gt_masks    = False,
    results_path    = results_400_path,
)
print(f'NIH results: {nih_results.shape}')

## 14. Summary Tables & Plots

In [ ]:
XAI_METHODS   = ['LayerCAM','FPN-LayerCAM','ScoreCAM','FPN-ScoreCAM','LIME']
METHOD_COLORS = ['#378ADD','#1D9E75','#D85A30','#7F77DD','#BA7517']


def summary_table(df, metrics, title):
    rows = []
    for m in XAI_METHODS:
        sub = df[df['method']==m]
        row = {'Method': m}
        for met in metrics:
            v = sub[met].dropna()
            row[met] = f'{v.mean():.4f} ± {v.std():.4f}' if len(v) else 'N/A'
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index('Method')
    print(f'\n{title}')
    display(tbl)
    return tbl


tbl_chex = summary_table(
    chex_results,
    ['pointing_game','iou_05','miou','deletion_auc','insertion_auc'],
    'CheXlocalize 413 — All metrics'
)
tbl_nih = summary_table(
    nih_results,
    ['deletion_auc','insertion_auc'],
    'ChestX-ray14 400 — Deletion & Insertion AUC'
)

tbl_chex.to_csv(EVAL_OUTPUT_DIR / 'summary_chexlocalize_413.csv')
tbl_nih.to_csv(EVAL_OUTPUT_DIR  / 'summary_chestxray14_400.csv')

# Per-label tables
for metric in ['pointing_game','iou_05','miou','deletion_auc','insertion_auc']:
    sub = chex_results[chex_results['gt_available']==True] \
          if metric in ['pointing_game','iou_05','miou'] else chex_results
    tbl = sub.groupby(['label','method'])[metric].mean().unstack('method')
    tbl = tbl[[m for m in XAI_METHODS if m in tbl.columns]]
    tbl.round(4).to_csv(EVAL_OUTPUT_DIR / f'per_label_{metric}_413.csv')

print('All tables saved.')

In [ ]:
# ── Bar charts: method comparison ─────────────────────────────────────────────
def plot_method_bars(df, metrics, titles, dataset, filepath):
    fig, axes = plt.subplots(1,len(metrics),figsize=(5*len(metrics),5))
    if len(metrics)==1: axes=[axes]
    cmap = dict(zip(XAI_METHODS,METHOD_COLORS))
    for ax,met,title in zip(axes,metrics,titles):
        means = [df[df['method']==m][met].dropna().mean() for m in XAI_METHODS]
        errs  = [df[df['method']==m][met].dropna().std()  for m in XAI_METHODS]
        bars  = ax.bar(XAI_METHODS,means,color=[cmap[m] for m in XAI_METHODS],
                       yerr=errs,capsize=4,edgecolor='black',linewidth=0.5)
        ax.set_title(title,fontsize=11); ax.set_ylabel('Score')
        ax.tick_params(axis='x',rotation=30); ax.grid(axis='y',alpha=0.3)
        for bar,v,e in zip(bars,means,errs):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+max(errs)*0.05+0.001,
                    f'{v:.3f}',ha='center',va='bottom',fontsize=8)
    plt.suptitle(f'{dataset}',fontsize=13)
    plt.tight_layout()
    plt.savefig(filepath,dpi=150,bbox_inches='tight')
    plt.show()
    print(f'Saved → {filepath}')


# Mask-based (CheXlocalize)
plot_method_bars(
    chex_results[chex_results['gt_available']==True],
    ['pointing_game','iou_05','miou'],
    ['Pointing Game ↑','IoU@0.5 ↑','mIoU ↑'],
    'CheXlocalize 413 — Mask-based metrics',
    EVAL_OUTPUT_DIR/'plot_mask_metrics_413.png'
)

# Del/Ins — both datasets
for ds_name,ds_df in [('CheXlocalize_413',chex_results),
                       ('ChestXray14_400',nih_results)]:
    plot_method_bars(
        ds_df,
        ['deletion_auc','insertion_auc'],
        ['Deletion AUC ↓','Insertion AUC ↑'],
        ds_name,
        EVAL_OUTPUT_DIR/f'plot_del_ins_{ds_name}.png'
    )

# ── Heatmaps: per-label × method ──────────────────────────────────────────────
for metric, cmap, title in [
    ('miou',          'YlGn',     'mIoU — labels × methods (CheXlocalize 413)'),
    ('pointing_game', 'YlGn',     'Pointing Game — labels × methods (CheXlocalize 413)'),
    ('deletion_auc',  'YlOrRd_r', 'Deletion AUC — labels × methods (CheXlocalize 413)'),
    ('insertion_auc', 'YlGn',     'Insertion AUC — labels × methods (CheXlocalize 413)'),
]:
    sub = chex_results[chex_results['gt_available']==True] \
          if metric in ['pointing_game','iou_05','miou'] else chex_results
    tbl = sub.groupby(['label','method'])[metric].mean().unstack('method')
    tbl = tbl[[m for m in XAI_METHODS if m in tbl.columns]].astype(float)
    fig, ax = plt.subplots(figsize=(len(tbl.columns)*1.6+2, len(tbl)*0.55+1.8))
    sns.heatmap(tbl,annot=True,fmt='.3f',cmap=cmap,linewidths=0.4,ax=ax,
                cbar_kws={'shrink':0.7},annot_kws={'size':9})
    ax.set_title(title,fontsize=11,pad=10)
    ax.set_xlabel('XAI Method'); ax.set_ylabel('Label')
    plt.tight_layout()
    out_fn = EVAL_OUTPUT_DIR / f'heatmap_{metric}_413.png'
    plt.savefig(out_fn,dpi=150,bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_fn}')

## 15. Final Summary

In [ ]:
print('='*65)
print('  Expanded XAI Evaluation — Final Summary')
print('='*65)
print(f'  CheXlocalize : {chex_results["filename"].nunique()} images evaluated')
print(f'  ChestX-ray14 : {nih_results["filename"].nunique()} images evaluated')
print()
print('── CheXlocalize (mask-based + mask-free) ──────────────────')
for m in XAI_METHODS:
    sub  = chex_results[chex_results['method']==m]
    mask = sub[sub['gt_available']==True]
    print(f'  {m:<16s}  '
          f'PG={mask["pointing_game"].mean():.3f}  '
          f'IoU={mask["iou_05"].mean():.3f}  '
          f'mIoU={mask["miou"].mean():.3f}  '
          f'Del={sub["deletion_auc"].mean():.3f}  '
          f'Ins={sub["insertion_auc"].mean():.3f}')
print()
print('── ChestX-ray14 (mask-free only) ──────────────────────────')
for m in XAI_METHODS:
    sub = nih_results[nih_results['method']==m]
    print(f'  {m:<16s}  '
          f'Del={sub["deletion_auc"].mean():.3f}  '
          f'Ins={sub["insertion_auc"].mean():.3f}')
print('='*65)